In [4]:
pip install transformers evaluate seqeval


In [5]:
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]


In [6]:
import pandas as pd

df = pd.read_csv("ner_dataset.csv")  # Your CSV file

# Convert to list of sentences and labels
sentences = df.groupby("sentence_id")["word"].apply(list).tolist()
labels = df.groupby("sentence_id")["label"].apply(list).tolist()


In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_and_align_labels(sentences, labels):
    tokenized_inputs = tokenizer(
        sentences,
        is_split_into_words=True,
        padding=True,
        truncation=True
    )

    all_labels = []
    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


In [21]:
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}


In [22]:
encodings = tokenize_and_align_labels(sentences, labels)

import torch

class NERDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

dataset = NERDataset(encodings)

# Split train/test
train_size = int(0.8 * len(dataset))
train_dataset = torch.utils.data.Subset(dataset, range(train_size))
eval_dataset = torch.utils.data.Subset(dataset, range(train_size, len(dataset)))


In [23]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list)
)
model.config.id2label = id2label
model.config.label2id = label2id


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [12]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',
    logging_steps=10,

)

metric = evaluate.load("seqeval")

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)

    true_preds = []
    true_labels = []

    for pred, lab in zip(preds, labels):
        p_list, l_list = [], []
        for p, l in zip(pred, lab):
            if l != -100:
                p_list.append(label_list[p])
                l_list.append(label_list[l])
        true_preds.append(p_list)
        true_labels.append(l_list)

    results = metric.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [15]:
sentence = "John works at Google in California"
tokens = sentence.split()
inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)
outputs = model(**inputs).logits
preds = np.argmax(outputs.detach().numpy(), axis=2)
pred_tags = [label_list[p] for p in preds[0][:len(tokens)]]
print("Tokens:", tokens)
print("Predicted Tags:", pred_tags)


Tokens: ['John', 'works', 'at', 'Google', 'in', 'California']
Predicted Tags: ['O', 'B-PER', 'O', 'O', 'B-ORG', 'O']


In [13]:
trainer.train()
print(trainer.evaluate())


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.721485
20,0.020425
30,0.003684
40,0.001993
50,0.001520
60,0.001299
70,0.001150
80,0.001061
90,0.001025
100,0.000941


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.00042909427429549396, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0, 'eval_runtime': 2.6234, 'eval_samples_per_second': 22.871, 'eval_steps_per_second': 3.05, 'epoch': 5.0}


In [14]:
sentence = "John works at Google in California"
tokens = sentence.split()
inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)
outputs = model(**inputs).logits
preds = np.argmax(outputs.detach().numpy(), axis=2)

pred_tags = [label_list[p] for p in preds[0][:len(tokens)]]
print("Tokens:", tokens)
print("Predicted Tags:", pred_tags)


Tokens: ['John', 'works', 'at', 'Google', 'in', 'California']
Predicted Tags: ['O', 'B-PER', 'O', 'O', 'B-ORG', 'O']


In [28]:
import warnings
import os
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=UserWarning, module="seqeval")
!pip install -q seqeval
import seqeval
warnings.filterwarnings("ignore", module=r"seqeval\..*")

In [29]:
import os
import logging
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()